# 01 — ADMM intuition

Why splitting works, on problems small enough to see every number.

No MPC yet, no graph yet. The goal is that by the end of this notebook the reader can predict, qualitatively, what `rho` does to the residuals before running anything.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from distributed_mpc_admm import plotting

plotting.apply_style("notebook")
rng = np.random.default_rng(0)

## The splitting idea

Start from `minimize f(x) + g(z) s.t. x = z`. Write the augmented Lagrangian, then the three ADMM steps. Show them as three lines of code on a scalar problem.

In [ ]:
# Scalar ADMM on f(x) = 0.5*(x-3)^2, g(z) = |z| (soft-threshold proximal step).
# Both sub-updates have closed forms, so every intermediate number is visible.

def soft_threshold(v, kappa):
    """Proximal operator of |.|: shrink v toward zero by kappa."""
    return np.sign(v) * np.maximum(np.abs(v) - kappa, 0.0)

def scalar_admm(rho, n_iter=10):
    x, z, lam = 0.0, 0.0, 0.0
    rows = []
    for k in range(n_iter):
        # x-update: minimise 0.5*(x-3)^2 + (rho/2)*(x - z + lam)^2  ->  closed form
        x = (3.0 + rho * (z - lam)) / (1.0 + rho)
        # z-update: soft threshold of (x + lam) at 1/rho
        z = soft_threshold(x + lam, 1.0 / rho)
        # dual update (scaled form)
        lam = lam + x - z
        rows.append((k, x, z, lam))
    return rows

rho = 1.0
print(f"{'k':>2} {'x':>10} {'z':>10} {'lam':>10}")
for k, x, z, lam in scalar_admm(rho):
    print(f"{k:>2} {x:10.6f} {z:10.6f} {lam:10.6f}")

# The problem is min 0.5*(x-3)^2 + |x|: f pulls toward 3, g pulls toward 0.
print("\nfixed point: x = z = 2, lam = 1  (the compromise that balances f' and the subgradient of g)")

## Two-agent averaging

The smallest consensus problem: two agents with different quadratic objectives that must agree on one number. Solve it by hand, then by ADMM.

In [ ]:
# Two-agent scalar consensus: agent i minimises 0.5*(x - a_i)^2, both must agree.
# The loop below IS consensus ADMM (Boyd et al. 2011, general form) -- written out
# explicitly so the reader sees all six lines. Do not import ConsensusADMM here.

a = np.array([-1.0, 3.0])      # the two agents' private targets
n = len(a)

def two_agent_admm(rho, n_iter=40):
    x = np.zeros(n)             # private variables
    lam = np.zeros(n)           # scaled duals, one per agent
    z = 0.0                     # the shared consensus value
    residual = []
    for _ in range(n_iter):
        # x-update: closed form for quadratic f_i
        x = (a + rho * (z - lam)) / (1.0 + rho)
        # z-update: plain average over all agents
        z = float(np.mean(x + lam))
        # dual update
        lam = lam + x - z
        residual.append(float(np.abs(x - z).max()))
    return x, z, lam, np.asarray(residual)

x, z, lam, residual = two_agent_admm(rho=1.0)
print(f"consensus value z = {z:.6f}   (analytic answer = mean of targets = {a.mean():.6f})")
print(f"private variables x = {np.round(x, 6)}")
print(f"max primal residual after 40 iters = {residual[-1]:.2e}")

## What rho actually does

Sweep rho over three decades and plot the residual traces on one axis. Large rho: primal residual falls fast, dual residual lags. Small rho: the reverse. This is the whole intuition behind residual balancing.

In [ ]:
import matplotlib.pyplot as plt

# Sweep rho over three decades and record primal/dual residual traces for the
# two-agent problem. Large rho: primal falls fast, dual lags. Small rho: the reverse.
# This is the whole intuition behind residual balancing.

rho_values = np.logspace(-2, 2, 9)

def run_sweep(rho, n_iter=200):
    x = np.zeros(n)
    lam = np.zeros(n)
    z = 0.0
    z_prev = 0.0
    primal, dual = [], []
    for _ in range(n_iter):
        x = (a + rho * (z - lam)) / (1.0 + rho)
        z = float(np.mean(x + lam))
        lam = lam + x - z
        primal.append(float(np.abs(x - z).max()))
        dual.append(rho * np.abs(z - z_prev))
        z_prev = z
    return np.asarray(primal), np.asarray(dual)

fig, ax = plt.subplots(figsize=(8, 5))
iters_to_conv = {}
for rho in rho_values:
    primal, dual = run_sweep(rho)
    (line,) = ax.semilogy(primal, linewidth=1.2, label=f"rho={rho:.2g} (primal)")
    ax.semilogy(dual, linestyle="--", color=line.get_color(), linewidth=1.0, alpha=0.8)
    converged = np.argwhere(primal <= 1e-6)
    iters_to_conv[rho] = int(converged[0, 0]) if converged.size else np.inf

best_rho = min(iters_to_conv, key=iters_to_conv.get)
ax.axvline(iters_to_conv[best_rho], color="tab:red", linestyle=":", alpha=0.6)
ax.annotate(
    f"best rho={best_rho:.2g} ({iters_to_conv[best_rho]} iters to 1e-6)",
    xy=(iters_to_conv[best_rho], 1e-6),
    xytext=(iters_to_conv[best_rho] + 8, 1e-2),
    arrowprops=dict(arrowstyle="->"),
)
ax.set_xlabel("ADMM iteration")
ax.set_ylabel("residual (log scale)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="lower left", fontsize="x-small")
ax.set_title("rho sweep: primal (solid) vs dual (dashed)")
plt.show()

## Over-relaxation

Add the `alpha` step and show the same problem with alpha = 1.0 vs 1.6.

In [ ]:
# Over-relaxation: in the z/dual updates, replace x with
#     x_hat = alpha * x + (1 - alpha) * z_old
# where z_old is the consensus value *before* the z-update. alpha=1.0 is plain ADMM;
# 1.5-1.8 typically buys 20-40% fewer iterations for convex consensus problems.

def two_agent_admm_relaxed(rho, alpha, n_iter=200):
    x = np.zeros(n)
    lam = np.zeros(n)
    z = 0.0
    primal = []
    for _ in range(n_iter):
        x = (a + rho * (z - lam)) / (1.0 + rho)
        x_hat = alpha * x + (1.0 - alpha) * z   # z here is the pre-update value
        z = float(np.mean(x_hat + lam))
        lam = lam + x_hat - z
        primal.append(float(np.abs(x - z).max()))
    return np.asarray(primal)

fig, ax = plt.subplots(figsize=(8, 5))
for alpha in (1.0, 1.5, 1.8):
    primal = two_agent_admm_relaxed(rho=1.0, alpha=alpha)
    converged = np.argwhere(primal <= 1e-6)
    iters = int(converged[0, 0]) if converged.size else np.inf
    ax.semilogy(primal, linewidth=1.2, label=f"alpha={alpha} ({iters} iters)")
ax.set_xlabel("ADMM iteration")
ax.set_ylabel("primal residual (log scale)")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
ax.set_title("Over-relaxation")
plt.show()

## From scalars to trajectories

The jump: replace the scalar `x` with a stacked input sequence and `g` with the coupling. Nothing about the three steps changes -- only the dimension.

In [ ]:
# The jump from scalars to trajectories changes only the dimension, not the three
# steps. Written out with the repository's block shapes so the reader can map the
# scalar loop above onto the real solver (see docs/README_math.md, section 5).

#   y_i^j   : (T, 2)   agent i's local copy of j's position trajectory
#   z^j     : (T, 2)   the consensus trajectory for agent j
#   lam_i^j : (T, 2)   scaled dual for the constraint y_i^j = z^j
#   U_i     : (T, 2)   agent i's input sequence (inside f_i)

# 1. x-update (one QP per agent, solved in parallel):
#    (U_i, {y_i^j}) <- argmin  f_i(U_i, {y_i^j})
#                      + (rho/2) * sum_{j in Ncl(i)} || y_i^j - z^j + lam_i^j ||_F^2
#
# 2. z-update (neighbor averaging, computed by agent j):
#    z^j <- (1 / |Ncl(j)|) * sum_{i in Ncl(j)} ( y_i^j + lam_i^j )
#
# 3. dual update:
#    lam_i^j <- lam_i^j + y_i^j - z^j

# Every term is a (T, 2) block. Nothing global enters: agent j only sums over the
# agents that hold a copy of *its* trajectory, i.e. its closed neighborhood.
print("symbolic restatement complete -- no numerics in this cell")

## Takeaway

The three ADMM steps never change; only what lives inside the norms does. `rho` trades primal against dual progress, which is exactly why residual balancing works.